In [2]:
DATA_PATH = r"data\synthetic_dataset.jsonl"

In [37]:
from google import genai
from google.genai import types
import wave
from loguru import logger


In [17]:
import json

text_data = []

with open(DATA_PATH, "r" , encoding="utf-8") as f:
    for line in f:
        text_data.append(json.loads(line))

print(text_data)

[{'id': '4080d11b-537b-4ecf-9971-a65129495e63', 'schema_version': '1.0', 'created_at': '2026-05-11T06:47:28.479139', 'status': 'generated', 'review_status': 'pending', 'source': 'gemini-2.5-flash-lite', 'speaker_id': '2', 'text': 'لو سمحت هو انا ممكن استلم الاوردر بتاعي من الفرع؟', 'emotion': 'speak naturally and calmly', 'speaker_style': 'use a casual conversational tone', 'speaking_rate': 'speaking_rate: normal', 'energy': 'with moderate energy', 'background_noise': 'clean audio', 'category': 'delivery', 'code_switching': False, 'contains_numbers': False, 'tags': ['delivery', 'pickup', 'order']}, {'id': 'df6e456d-62a9-40c7-8ee4-6ee534c47015', 'schema_version': '1.0', 'created_at': '2026-05-11T06:48:54.313810', 'status': 'generated', 'review_status': 'pending', 'source': 'gemini-2.5-flash-lite', 'speaker_id': '1', 'text': 'يا باشا هو الطلب وصل ولا لسه؟', 'emotion': 'speak naturally and calmly', 'speaker_style': 'speak like a natural phone conversation', 'speaking_rate': 'speaking_rate

In [18]:
text_data[1]

{'id': 'df6e456d-62a9-40c7-8ee4-6ee534c47015',
 'schema_version': '1.0',
 'created_at': '2026-05-11T06:48:54.313810',
 'status': 'generated',
 'review_status': 'pending',
 'source': 'gemini-2.5-flash-lite',
 'speaker_id': '1',
 'text': 'يا باشا هو الطلب وصل ولا لسه؟',
 'emotion': 'speak naturally and calmly',
 'speaker_style': 'speak like a natural phone conversation',
 'speaking_rate': 'speaking_rate: normal',
 'energy': 'with moderate energy',
 'background_noise': 'clean audio',
 'category': 'delivery',
 'code_switching': False,
 'contains_numbers': False,
 'tags': ['delivery', 'order status', 'waiting']}

# utils

In [8]:
# =========================
# Save wav helper
# =========================
def wave_file(filename, pcm, channels=1, rate=24000, sample_width=2):
    with wave.open(filename, "wb") as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(sample_width)
        wf.setframerate(rate)
        wf.writeframes(pcm)

In [11]:
# =========================
# Convert JSON -> TTS Prompt
# =========================
def build_tts_prompt(data: dict) -> str:

    

    emotion = data["emotion"]
    style = data["speaker_style"]
    rate = data["speaking_rate"]
    energy = data["energy"]

    text = data["text"]

    prompt = f"""
{emotion}, {style}, {rate}, {energy}.

Say in Egyption Arabic:
{text}
"""

    return prompt.strip()


In [32]:
# =========================
# Build Prompt
# =========================
tts_prompt = build_tts_prompt(text_data[10])

print(tts_prompt)

speak naturally and calmly, use a casual conversational tone, speaking_rate: normal, with moderate energy.

Say in Egyption Arabic:
يا دكتور هو المنهج بتاع الترم التاني هينزل امتى بالظبط؟


In [33]:
text_data[10]['speaker_id']

'4'

# tts generator  

In [24]:
client = genai.Client()

speaker_id_map = {
    "1" : "Zephyr",
    "2" : "Kore" , 
    "3" : "Algenib", 
    "4" : "Algieba"
}

speaker_id_map[text_data[2]['speaker_id']]

'Laomedeia'

In [36]:


response = client.models.generate_content(
    model="gemini-2.5-flash-preview-tts",
    contents=tts_prompt,
    config=types.GenerateContentConfig(
        response_modalities=["AUDIO"],
        speech_config=types.SpeechConfig(
            voice_config=types.VoiceConfig(
                prebuilt_voice_config=types.PrebuiltVoiceConfig(
                    voice_name=speaker_id_map[text_data[2]['speaker_id']],
                )
            )
        ),
    )
)

data = response.candidates[0].content.parts[0].inline_data.data

file_name = "out.wav"
wave_file(file_name, data)

print(f"Saved to {file_name}")

Saved to out.wav


# class 

In [ ]:
from pydantic import BaseModel


class AudioGenerationResult(BaseModel):

    audio_id: str
    prompt_id: str

    audio_path: str

    text : str

    voice_name: str
    speaker_id: str
    tts_model: str

    background_noise :str

    sample_rate: int

    status: str
    review_status : str

In [62]:
class TTSPromptBuilder:

    @staticmethod
    def build(data: dict) -> str:

        emotion = data["emotion"]
        style = data["speaker_style"]
        rate = data["speaking_rate"]
        energy = data["energy"]

        text = data["text"]

        prompt = f"""
{emotion}, {style}, {rate}, {energy}.

Say in Egyptian Arabic:
{text}
"""

        return prompt.strip()

In [ ]:
def save_to_jsonl(OUTPUT_FILE , record: dict):
    with open(OUTPUT_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")

In [90]:
from pathlib import Path

In [95]:
p = "data\synthetic_audio_dataset.jsonl"

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
C:\Users\DELL\AppData\Local\Temp\ipykernel_12648\3781049386.py:1: SyntaxWarning: invalid escape sequence '\s'
  p = "data\synthetic_audio_dataset.jsonl"


In [96]:
Path(p)

WindowsPath('data/synthetic_audio_dataset.jsonl')

In [ ]:
import asyncio
import json


from datetime import datetime

from loguru import logger
from google.genai import types


TTS_MODEL_NAME = "gemini-2.5-flash-preview-tts"


class AudioGenerationTTSService:

    def __init__(
        self,
        client,
        tts_model_name="gemini-2.5-flash-preview-tts",
        output_dir="audio_outputs",
        jsonl_path="data/synthetic_audio_dataset.jsonl",
        max_concurrent_tasks=5
    ):

        self.client = client

        self.tts_model_name = tts_model_name

        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        self.jsonl_path = Path(jsonl_path)
        self.jsonl_path.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        # concurrency control
        self.semaphore = asyncio.Semaphore(
            max_concurrent_tasks
        )

        self.speaker_id_map = {
            "1": "Zephyr",
            "2": "Kore",
            "3": "Algenib",
            "4": "Algieba"
        }

        logger.info("✅ GeminiTTSService initialized")

    # =====================================================
    # Save JSONL
    # =====================================================

    def save_to_jsonl(self, record: dict):

        with open(
            self.jsonl_path,
            "a",
            encoding="utf-8"
        ) as f:

            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                    default=str
                ) + "\n"
            )

    # =====================================================
    # Generate ONE audio sample
    # =====================================================

    async def generate_audio(
        self,
        prompt_record: dict
    ):

        async with self.semaphore:

            start_time = datetime.utcnow()

            try:

                logger.info(
                    f"🎤 Generating audio | "
                    f"id={prompt_record['id']}"
                )

                audio_id = prompt_record["id"]

                output_path = (
                    self.output_dir /
                    f"{audio_id}.wav"
                )

                # ======================================
                # Caching / Skip Existing
                # ======================================

                if output_path.exists():

                    logger.info(
                        f"⏭️ Skipping existing audio | "
                        f"id={audio_id}"
                    )

                    return None

                # ======================================
                # Build TTS Prompt
                # ======================================

                tts_prompt = TTSPromptBuilder.build(
                    prompt_record
                )

                # ======================================
                # Select Voice
                # ======================================

                speaker_id = prompt_record["speaker_id"]

                voice_name = self.speaker_id_map[
                    speaker_id
                ]

                # ======================================
                # Generate Audio
                # Run blocking API in thread
                # ======================================

                response = await asyncio.to_thread(
                    self.client.models.generate_content,

                    model=self.tts_model_name,

                    contents=tts_prompt,

                    config=types.GenerateContentConfig(
                        response_modalities=["AUDIO"],

                        speech_config=types.SpeechConfig(
                            voice_config=types.VoiceConfig(
                                prebuilt_voice_config=(
                                    types.PrebuiltVoiceConfig(
                                        voice_name=voice_name
                                    )
                                )
                            )
                        ),
                    )
                )

                # ======================================
                # Extract PCM
                # ======================================

                pcm_data = (
                    response
                    .candidates[0]
                    .content.parts[0]
                    .inline_data.data
                )

                # ======================================
                # Save WAV
                # ======================================

                await asyncio.to_thread(
                    wave_file,
                    str(output_path),
                    pcm_data
                )

                # ======================================
                # Build Result Schema
                # ======================================

                result = AudioGenerationResult(

                    audio_id=audio_id,

                    prompt_id=prompt_record["id"],

                    audio_path=str(output_path),

                    text=prompt_record["text"],

                    background_noise=prompt_record[
                        "background_noise"
                    ],

                    voice_name=voice_name,

                    speaker_id=speaker_id,

                    tts_model=self.tts_model_name,

                    sample_rate=24000,

                    status="generated",

                    review_status="pending"
                )

                # ======================================
                # Save Metadata JSONL
                # ======================================

                self.save_to_jsonl(
                    result.model_dump()
                )

                duration = (
                    datetime.utcnow() - start_time
                ).total_seconds()

                logger.success(
                    f"✅ Audio generated | "
                    f"id={audio_id} | "
                    f"time={duration:.2f}s"
                )

                return result

            except Exception as e:

                logger.error(
                    f"❌ Audio generation failed | "
                    f"id={prompt_record.get('id')} | "
                    f"error={str(e)}"
                )

                return None

    # =====================================================
    # Generate Multiple Audio Samples in Parallel
    # =====================================================

    async def generate_parallel(
        self,
        prompt_records: list[dict]
    ):

        logger.info(
            f"🚀 Starting parallel TTS generation | "
            f"samples={len(prompt_records)}"
        )

        tasks = [
            self.generate_audio(record)
            for record in prompt_records
        ]

        results = await asyncio.gather(*tasks)

        # remove failed/skipped
        results = [
            r for r in results
            if r is not None
        ]

        logger.success(
            f"🏁 Parallel TTS completed | "
            f"generated={len(results)}"
        )

        return results

In [87]:
text_data[0]

{'id': '4080d11b-537b-4ecf-9971-a65129495e63',
 'schema_version': '1.0',
 'created_at': '2026-05-11T06:47:28.479139',
 'status': 'generated',
 'review_status': 'pending',
 'source': 'gemini-2.5-flash-lite',
 'speaker_id': '2',
 'text': 'لو سمحت هو انا ممكن استلم الاوردر بتاعي من الفرع؟',
 'emotion': 'speak naturally and calmly',
 'speaker_style': 'use a casual conversational tone',
 'speaking_rate': 'speaking_rate: normal',
 'energy': 'with moderate energy',
 'background_noise': 'clean audio',
 'category': 'delivery',
 'code_switching': False,
 'contains_numbers': False,
 'tags': ['delivery', 'pickup', 'order']}

In [89]:
tts_service = GeminiTTSService(client)

results = await tts_service.generate_parallel(
    text_data[1:5]
)

2026-05-11 13:33:02.278 | INFO     | __main__:__init__:53 - ✅ GeminiTTSService initialized
2026-05-11 13:33:02.283 | INFO     | __main__:generate_parallel:251 - 🚀 Starting parallel TTS generation | samples=4
C:\Users\DELL\AppData\Local\Temp\ipykernel_12648\1591100779.py:86: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = datetime.utcnow()
2026-05-11 13:33:02.284 | INFO     | __main__:generate_audio:90 - 🎤 Generating audio | id=df6e456d-62a9-40c7-8ee4-6ee534c47015
2026-05-11 13:33:02.285 | INFO     | __main__:generate_audio:108 - ⏭️ Skipping existing audio | id=df6e456d-62a9-40c7-8ee4-6ee534c47015
2026-05-11 13:33:02.285 | INFO     | __main__:generate_audio:90 - 🎤 Generating audio | id=1a21ae12-1078-4e60-96fb-b94dee8a3235
2026-05-11 13:33:02.287 | INFO     | __main__:generate_audio:108 - ⏭️ Skipping existing audio | id=

In [60]:
res.model_dump()

{'audio_id': 'a0fc2ea2-1c9f-434c-8e2e-2c862a570816',
 'prompt_id': 'df6e456d-62a9-40c7-8ee4-6ee534c47015',
 'audio_path': 'audio_outputs\\a0fc2ea2-1c9f-434c-8e2e-2c862a570816.wav',
 'text': 'يا باشا هو الطلب وصل ولا لسه؟',
 'voice_name': 'Zephyr',
 'speaker_id': '1',
 'tts_model': 'gemini-2.5-flash-preview-tts',
 'sample_rate': 24000,
 'status': 'generated',
 'review_status': 'pending'}